# 🌍 Phase 2 — Public Real-World Benchmark: WebNLG (Human-Annotated)
## Moving Beyond Synthetic Data: Testing on Real Human-Written Sentences

### 🎯 Objective
Our previous evaluations used synthetic data:
- **Clean Laboratory Test**: 100.00% Macro F1
- **Adversarial Stress Test**: 61.67% Macro F1 (exposed positional bias & negative suppression)

Now we evaluate on the **Gold Standard Public Benchmark: WebNLG** (100 human-annotated sentences from DBpedia / Wikipedia spanning airports, buildings, monuments, written works, and sports teams).

We test two models head-to-head:
1. **Untuned Base Model (`Qwen2.5-1.5B-Instruct`)**: How well does the base model handle real human-written text zero-shot?
2. **Track 1 Fine-Tuned Adapter (`Qwen-1.5B-Legal-Graph-Adapter`)**: Did fine-tuning on corporate litigation generalize or did it cause catastrophic forgetting on open-domain human text?

---
### 🛠️ Hardware & Setup
- **Target Runtime**: Google Colab T4 GPU (or local GPU)
- **Model**: `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit NF4
- **Execution Time**: ~8–12 minutes total

### Step 1: Install Dependencies & Hardware Audit

In [ ]:
# Run in Google Colab:
!pip install -q transformers peft bitsandbytes accelerate tqdm

In [ ]:
import sys
import os
import json
import re
import time
import torch
import transformers
from tqdm import tqdm

print("=" * 65)
print("🚀 WEBNLG REAL-WORLD BENCHMARK — HARDWARE AUDIT")
print("=" * 65)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("GPU Device Name      : CPU Mode (Quantization disabled)")
print(f"Transformers Version : {transformers.__version__}")
print("=" * 65)

### Step 2: Locate WebNLG Benchmark Data & Adapter Directory

In [ ]:
# Ensure repository is present
if not os.path.exists("llm-graph-ontology"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git
    if os.path.exists("Fine-Tuning-PoCs/llm-graph-ontology"):
        %cd Fine-Tuning-PoCs

def find_path(rel_path):
    candidates = [
        rel_path,
        f"../{rel_path}",
        f"llm-graph-ontology/{rel_path}",
        f"Fine-Tuning-PoCs/llm-graph-ontology/{rel_path}",
        f"/content/Fine-Tuning-PoCs/llm-graph-ontology/{rel_path}",
        f"/content/llm-graph-ontology/{rel_path}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Cannot find path: {rel_path}")

webnlg_file = find_path("data/benchmark/webnlg_human_benchmark_100.json")
adapter_dir = find_path("models/qwen-1.5b-legal-graph-adapter")

with open(webnlg_file, "r", encoding="utf-8") as f:
    webnlg_samples = json.load(f)

print(f"✅ Loaded {len(webnlg_samples)} human-annotated WebNLG samples from: {webnlg_file}")
print(f"✅ Located adapter weights at: {adapter_dir}")

### Step 3: Load Untuned Base Model in 4-bit NF4

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading base tokenizer & model: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        device_map="auto"
    )

base_model.eval()
print("✅ Base Model Loaded in 4-bit NF4.")

### Step 4: Define Extraction Function & JSON Parser

In [ ]:
def extract_triples_from_model(current_model, text_passage):
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a knowledge graph triplet extractor. "
                "Extract factual triples from the input text representing all entities and relationships. "
                "Output ONLY a valid JSON list of objects with keys: 'subject', 'predicate', 'object'."
            )
        },
        {
            "role": "user",
            "content": f"Extract knowledge graph triples from this text:\n{text_passage}"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(current_model.device)
    with torch.no_grad():
        out = current_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def clean_str(s):
    return re.sub(r'[\W_]+', ' ', s.strip().lower()).strip()

def parse_triples_json(raw_text):
    cleaned = re.sub(r'```(?:json)?', '', raw_text).strip('` \n')
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return set(
                (clean_str(item.get('subject', '')), clean_str(item.get('predicate', '')), clean_str(item.get('object', '')))
                for item in data if isinstance(item, dict)
            )
    except Exception:
        pass
    return set()

print("Extraction and evaluation parser ready.")

### Step 5: Evaluate Untuned Base Model on WebNLG (100 Human Samples)

In [ ]:
print("=" * 65)
print("🔍 EVALUATING UNTUNED BASE MODEL ON WEBNLG (100 SAMPLES)")
print("=" * 65)

base_prec_list, base_rec_list, base_f1_list = [], [], []
start_time = time.time()

for item in tqdm(webnlg_samples, desc="Base Model WebNLG"):
    raw_out = extract_triples_from_model(base_model, item['input_text'])
    pred_set = parse_triples_json(raw_out)
    target_set = set(
        (clean_str(t['subject']), clean_str(t['predicate']), clean_str(t['object']))
        for t in item['target_triples']
    )
    
    tp = len(pred_set.intersection(target_set))
    prec = tp / len(pred_set) if len(pred_set) > 0 else 0.0
    rec = tp / len(target_set) if len(target_set) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    base_prec_list.append(prec)
    base_rec_list.append(rec)
    base_f1_list.append(f1)

base_avg_prec = sum(base_prec_list) / len(base_prec_list)
base_avg_rec = sum(base_rec_list) / len(base_rec_list)
base_avg_f1 = sum(base_f1_list) / len(base_f1_list)

print(f"\nBase Model Evaluation Complete in {(time.time() - start_time)/60:.2f} minutes!")
print(f"Base Precision : {base_avg_prec*100:.2f}%")
print(f"Base Recall    : {base_avg_rec*100:.2f}%")
print(f"Base Macro F1  : {base_avg_f1*100:.2f}%")

### Step 6: Attach Fine-Tuned Adapter & Evaluate on WebNLG (100 Human Samples)

In [ ]:
print("\n" + "=" * 65)
print("🔍 ATTACHING FINE-TUNED ADAPTER & EVALUATING ON WEBNLG")
print("=" * 65)

tuned_model = PeftModel.from_pretrained(base_model, adapter_dir)
tuned_model.eval()

tuned_prec_list, tuned_rec_list, tuned_f1_list = [], [], []
start_time = time.time()

for item in tqdm(webnlg_samples, desc="Tuned Model WebNLG"):
    raw_out = extract_triples_from_model(tuned_model, item['input_text'])
    pred_set = parse_triples_json(raw_out)
    target_set = set(
        (clean_str(t['subject']), clean_str(t['predicate']), clean_str(t['object']))
        for t in item['target_triples']
    )
    
    tp = len(pred_set.intersection(target_set))
    prec = tp / len(pred_set) if len(pred_set) > 0 else 0.0
    rec = tp / len(target_set) if len(target_set) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    tuned_prec_list.append(prec)
    tuned_rec_list.append(rec)
    tuned_f1_list.append(f1)

tuned_avg_prec = sum(tuned_prec_list) / len(tuned_prec_list)
tuned_avg_rec = sum(tuned_rec_list) / len(tuned_rec_list)
tuned_avg_f1 = sum(tuned_f1_list) / len(tuned_f1_list)

print(f"\nTuned Model Evaluation Complete in {(time.time() - start_time)/60:.2f} minutes!")
print(f"Tuned Precision : {tuned_avg_prec*100:.2f}%")
print(f"Tuned Recall    : {tuned_avg_rec*100:.2f}%")
print(f"Tuned Macro F1  : {tuned_avg_f1*100:.2f}%")

### Step 7: Print Master 3-Way Benchmark Comparison Table

In [ ]:
print("\n" + "=" * 80)
print("📊 MASTER MULTI-DATASET BENCHMARK SCORECARD: SYNTHETIC VS. HUMAN WEBNLG")
print("=" * 80)
print(f"{'Evaluation Setting / Dataset':<42} | {'Macro F1':<10} | {'Status / Context'}")
print("-" * 80)
print(f"{'1. Clean Synthetic Benchmark (Notebook 02)':<42} | {'100.00%':<10} | Laboratory In-Domain")
print(f"{'2. Adversarial Synthetic Stress Test (Notebook 03)':<42} | {'61.67%':<10} | Red-Team Failure Boundary")
print(f"{'3. WebNLG Real Human Text (Base Model Zero-Shot)':<42} | {f'{base_avg_f1*100:.2f}%':<10} | Real-World Human Benchmark")
print(f"{'4. WebNLG Real Human Text (Fine-Tuned Adapter)':<42} | {f'{tuned_avg_f1*100:.2f}%':<10} | Out-Of-Domain Transfer")
print("=" * 80)

webnlg_results = {
    "benchmark": "WebNLG_v1_Human_Annotated",
    "total_samples": len(webnlg_samples),
    "base_model": {
        "precision": round(base_avg_prec, 4),
        "recall": round(base_avg_rec, 4),
        "f1": round(base_avg_f1, 4)
    },
    "finetuned_adapter": {
        "precision": round(tuned_avg_prec, 4),
        "recall": round(tuned_avg_rec, 4),
        "f1": round(tuned_avg_f1, 4)
    },
    "transfer_delta": round(tuned_avg_f1 - base_avg_f1, 4)
}

with open("webnlg_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(webnlg_results, f, indent=2)
print("Saved WebNLG metrics to webnlg_benchmark_results.json")